In [ ]:
# !pip install yfinance
# !pip install TA-Lib 
# !pip install numpy
# !pip install pandas
# !pip install vectorbt
# !pip install scipy

In [ ]:
# Clone repo for lib/ access (Colab only)
import os
if not os.path.exists('/content/trading-strategies'):
    !git clone https://github.com/r-giov/trading-strategies.git /content/trading-strategies
os.chdir('/content/trading-strategies')

In [ ]:
import sys, os
# Ensure lib/ is importable (for Google Sheets logging)
for _p in ['/content/trading-strategies', os.path.join(os.getcwd(), '..')]:
    if os.path.isdir(os.path.join(_p, 'lib')) and _p not in sys.path:
        sys.path.insert(0, _p)

import yfinance as yf
import talib
import numpy as np
import pandas as pd
import vectorbt as vbt
import warnings
from scipy import stats
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
pd.set_option('future.no_silent_downcasting', True)
from lib.data_manager import download, download_multi, setup_colab_secrets

In [ ]:
# Data: yfinance (daily) | Alpaca (intraday) — auto-selected by data_manager
# For Colab: run setup_colab_secrets() if using Alpaca intraday features
# DOWNLOAD STOCK DATA
TICKER = 'SPY'
START_DATE = '2018-01-01'

stock_data = download(TICKER, START_DATE)

if not stock_data.empty:
    print(f"Successfully downloaded {len(stock_data)} records for {TICKER} from {START_DATE}")
    print(f"Data range: {stock_data.index.min().date()} to {stock_data.index.max().date()}")
    print("\nFirst 5 rows:")
    print(stock_data.head())
else:
    print(f"Failed to download {TICKER} data from yfinance")
stock_data

In [ ]:
# TECHNICAL ANALYSIS INDICATORS USING TA-LIB

if "stock_data" not in locals():
    raise ValueError("Please run the stock data download cell first")

if isinstance(stock_data.columns, pd.MultiIndex):
    close = stock_data[("Close", TICKER)].values
    high = stock_data[("High", TICKER)].values
    low = stock_data[("Low", TICKER)].values
    open_ = stock_data[("Open", TICKER)].values
    volume = stock_data[("Volume", TICKER)].values
else:
    close = stock_data["Close"].values
    high = stock_data["High"].values
    low = stock_data["Low"].values
    open_ = stock_data["Open"].values
    volume = stock_data["Volume"].values

prices = close.astype(float)

# Standard indicators
SMA_20  = talib.SMA(prices, timeperiod=20)
SMA_50  = talib.SMA(prices, timeperiod=50)
EMA_12  = talib.EMA(prices, timeperiod=12)
EMA_26  = talib.EMA(prices, timeperiod=26)
MACD_line, Signal_line, MACD_hist = talib.MACD(prices, fastperiod=12, slowperiod=26, signalperiod=9)
RSI_14  = talib.RSI(prices, timeperiod=14)

# Bollinger Bands (default 20, 2.0)
BB_UPPER, BB_MIDDLE, BB_LOWER = talib.BBANDS(prices, timeperiod=20, nbdevup=2.0, nbdevdn=2.0, matype=0)

indicators_df = pd.DataFrame({
    'Close': prices, 'SMA_20': SMA_20, 'SMA_50': SMA_50,
    'EMA_12': EMA_12, 'EMA_26': EMA_26,
    'MACD': MACD_line, 'Signal': Signal_line, 'RSI_14': RSI_14,
    'BB_Upper': BB_UPPER, 'BB_Middle': BB_MIDDLE, 'BB_Lower': BB_LOWER
}, index=stock_data.index)

print(f"Indicators computed for {TICKER}")
print(f"Bollinger Bands: Upper/Middle/Lower (20, 2.0 std)")
indicators_df.tail(5)

In [ ]:
# PREPARE PRICE SERIES

warnings.filterwarnings("ignore", message="Degrees of freedom <= 0 for slice", category=RuntimeWarning)
warnings.filterwarnings("ignore", message="invalid value encountered in scalar divide", category=RuntimeWarning)

def select_close_series(df, ticker):
    if isinstance(df.columns, pd.MultiIndex):
        if ('Close', ticker) in df.columns:
            s = df[('Close', ticker)]
        else:
            cols = [c for c in df.columns if 'Close' in str(c)]
            if not cols:
                raise KeyError("Close not found")
            s = df[cols[0]]
    else:
        s = df['Close']
    return s.astype(float).squeeze()

close = select_close_series(stock_data, TICKER)
close.name = 'price'

TRAIN_RATIO = 0.60
split_idx = int(len(close) * TRAIN_RATIO)
train_close = close.iloc[:split_idx].copy()
val_close   = close.iloc[split_idx:].copy()

print(f"Data ready: train={train_close.index[0].date()} to {train_close.index[-1].date()} | val={val_close.index[0].date()} to {val_close.index[-1].date()}")

BOLLINGER BAND MEAN REVERSION GRID SEARCH - TRAINING SET
---------------------------------------------------------

**Strategy**: Buy when price crosses BELOW the lower Bollinger Band (oversold).
Exit when price crosses ABOVE the middle band (mean reversion target) or upper band.

**Signal Logic** (1-bar execution delay):
- Entry: close crosses below lower BB (price < lower_band AND prev_close >= prev_lower_band)
- Exit Type 0: close crosses above middle BB (mean reversion to average)
- Exit Type 1: close crosses above upper BB (full reversion + overshoot)
- All signals shifted 1 bar for execution delay

**Parameters**: bb_period, bb_std, exit_type

In [ ]:
# Define Parameter Ranges for Bollinger Band Mean Reversion

bb_period_range = [10, 15, 20, 25, 30, 35, 40]
bb_std_range = [1.5, 1.75, 2.0, 2.25, 2.5, 2.75, 3.0]
exit_type_range = [0, 1]  # 0 = middle band exit, 1 = upper band exit

print("Bollinger Band Period (lookback):")
for i, p in enumerate(bb_period_range, 1):
    print(f"  {i}. {p} periods")

print("\nBollinger Band Std Dev (width):")
for i, s in enumerate(bb_std_range, 1):
    print(f"  {i}. {s} std devs")

print("\nExit Type:")
print("  0 = Middle Band (conservative mean reversion)")
print("  1 = Upper Band (full reversion + overshoot)")

bb_combinations = []
for period in bb_period_range:
    for std in bb_std_range:
        for exit_t in exit_type_range:
            bb_combinations.append((period, std, exit_t))

print(f"\nGenerated {len(bb_combinations)} Bollinger Band combinations")
print("\nFirst 10 combinations preview:")
for i, (p, s, e) in enumerate(bb_combinations[:10], 1):
    exit_label = "Middle" if e == 0 else "Upper"
    print(f"  {i:2d}. Period: {p} | Std: {s} | Exit: {exit_label}")
if len(bb_combinations) > 10:
    print(f"   ... and {len(bb_combinations) - 10} more combinations")

print("\nReady to test all combinations on training data!")

In [ ]:
# Initialize Bollinger Band Results Collection System

grid_search_results = []

print("Bollinger Band Mean Reversion Results Collection System Initialized")
print(f"   - Will test {len(bb_combinations)} BB parameter combinations")
print("   - Results will be stored in 'grid_search_results' list")

metrics_to_collect = [
    "bb_period", "bb_std", "exit_type",
    "total_return", "annualized_return",
    "sharpe_ratio", "sortino_ratio", "calmar_ratio", "omega_ratio",
    "information_ratio", "tail_ratio", "deflated_sharpe_ratio",
    "max_drawdown", "volatility", "ulcer_index",
    "win_rate", "total_trades", "avg_trade_duration", "expectancy",
    "profit_factor", "sqn",
    "payoff_ratio", "largest_win", "largest_loss",
    "avg_win_amount", "avg_loss_amount", "winning_streak", "losing_streak",
    "recovery_factor", "gain_to_pain_ratio", "serenity_index"
]

print("Metrics to collect for each BB combination:")
for i, metric in enumerate(metrics_to_collect, 1):
    print(f"  {i}. {metric.replace('_', ' ').title()}")

print("Ready to start the Bollinger Band grid search!")

In [ ]:
# BOLLINGER BAND MEAN REVERSION GRID SEARCH ON TRAINING DATA

print("INITIATING BOLLINGER BAND MEAN REVERSION GRID SEARCH OPTIMIZATION")
print("=" * 70)
print(f"Testing Strategy: Bollinger Band Mean Reversion")
print(f"Training Period: {train_close.index[0].date()} to {train_close.index[-1].date()}")
print(f"Initial Capital: $100,000")
print(f"Transaction Costs: 0.05% per trade (fees + slippage)")
print(f"Optimization Metric: Sharpe Ratio (risk-adjusted returns)")
print("=" * 70)

total_combinations = len(bb_combinations)
print(f"Total combinations to test: {total_combinations}")
print(f"Processing sequentially with progress every 50 combos\n")

grid_search_results = []
successful_tests = 0
failed_tests = 0
skipped_low_trades = 0

train_close_vals = train_close.values.astype(float)
train_idx = train_close.index
years = max((train_idx[-1] - train_idx[0]).days / 365.25, 1e-9)

for combo_num, (bb_period, bb_std, exit_type) in enumerate(bb_combinations, 1):
    try:
        bb_upper, bb_middle, bb_lower = talib.BBANDS(train_close_vals, timeperiod=bb_period, nbdevup=bb_std, nbdevdn=bb_std, matype=0)
        
        close_s = pd.Series(train_close_vals, index=train_idx)
        bb_upper_s = pd.Series(bb_upper, index=train_idx)
        bb_middle_s = pd.Series(bb_middle, index=train_idx)
        bb_lower_s = pd.Series(bb_lower, index=train_idx)

        # Entry: close crosses below lower band
        entries_raw = (close_s.shift(1) >= bb_lower_s.shift(1)) & (close_s < bb_lower_s)
        
        # Exit: depends on exit_type
        if exit_type == 0:
            exits_raw = (close_s.shift(1) <= bb_middle_s.shift(1)) & (close_s > bb_middle_s)
        else:
            exits_raw = (close_s.shift(1) <= bb_upper_s.shift(1)) & (close_s > bb_upper_s)

        # 1-bar execution delay
        entries_shifted = entries_raw.shift(1)
        entries = pd.Series(np.where(entries_shifted.isna(), False, entries_shifted), index=train_idx, dtype=bool)
        exits_shifted = exits_raw.shift(1)
        exits = pd.Series(np.where(exits_shifted.isna(), False, exits_shifted), index=train_idx, dtype=bool)

        pf = vbt.Portfolio.from_signals(close=train_close, entries=entries, exits=exits,
                                        init_cash=100_000, fees=0.0005, slippage=0.0005, freq='D')
        trades = pf.trades
        total_trades = len(trades)

        if total_trades < 10:
            skipped_low_trades += 1
            continue

        trades_per_year = total_trades / years
        total_return = float(pf.total_return())
        annualized_return = float(pf.annualized_return(freq='D'))
        max_drawdown = float(pf.max_drawdown())
        volatility = float(pf.annualized_volatility(freq='D'))
        sharpe_ratio = float(pf.sharpe_ratio(freq='D'))
        sortino_ratio = float(pf.sortino_ratio(freq='D'))
        calmar_ratio = annualized_return / abs(max_drawdown) if abs(max_drawdown) > 1e-9 else np.nan

        win_rate_pct = np.nan; profit_factor = np.nan; expectancy = 0.0
        avg_win_amount = 0.0; avg_loss_amount = 0.0; largest_win = 0.0; largest_loss = 0.0
        winning_streak = np.nan; losing_streak = np.nan; avg_trade_duration = np.nan; sqn = np.nan

        tr = trades.returns.values if hasattr(trades.returns, 'values') else np.array(trades.returns)
        if tr.size > 0:
            pos = tr[tr > 0]; neg = tr[tr < 0]
            win_rate_pct = (len(pos) / len(tr)) * 100.0 if len(tr) > 0 else np.nan
            gains = pos.sum() if len(pos) else 0.0
            losses = abs(neg.sum()) if len(neg) else 0.0
            profit_factor = gains / losses if losses > 0 else np.inf
            expectancy = float(tr.mean())
            avg_win_amount = float(pos.mean()) if len(pos) else 0.0
            avg_loss_amount = float(abs(neg.mean())) if len(neg) else 0.0
            largest_win = float(pos.max()) if len(pos) else 0.0
            largest_loss = float(neg.min()) if len(neg) else 0.0
            sqn = float(tr.mean() / tr.std() * np.sqrt(len(tr))) if tr.std() > 0 else np.nan
            try:
                winning_streak = int(trades.winning_streak())
                losing_streak = int(trades.losing_streak())
            except: pass

        returns = pf.returns()
        cum = (1 + returns).cumprod(); peak = cum.cummax(); dd = (cum - peak) / peak
        ulcer_index = float(np.sqrt((dd.pow(2)).mean())) if len(dd) > 0 else np.nan
        payoff_ratio = (avg_win_amount / avg_loss_amount) if avg_loss_amount > 0 else np.inf
        rets = returns.values
        omega_ratio = float(rets[rets > 0].sum() / abs(rets[rets < 0].sum())) if abs(rets[rets < 0].sum()) > 0 else np.inf
        recovery_factor = total_return / abs(max_drawdown) if abs(max_drawdown) > 1e-9 else np.nan
        gain_to_pain = float(rets.sum() / abs(rets[rets < 0].sum())) if abs(rets[rets < 0].sum()) > 0 else np.inf
        serenity_index = recovery_factor / ulcer_index if ulcer_index and ulcer_index > 0 else np.nan

        grid_search_results.append({
            "bb_period": bb_period, "bb_std": bb_std, "exit_type": exit_type,
            "total_return": total_return, "annualized_return": annualized_return,
            "sharpe_ratio": sharpe_ratio, "sortino_ratio": sortino_ratio,
            "calmar_ratio": calmar_ratio, "omega_ratio": omega_ratio,
            "max_drawdown": max_drawdown, "volatility": volatility, "ulcer_index": ulcer_index,
            "win_rate": win_rate_pct, "total_trades": total_trades,
            "expectancy": expectancy, "profit_factor": profit_factor, "sqn": sqn,
            "payoff_ratio": payoff_ratio, "largest_win": largest_win, "largest_loss": largest_loss,
            "avg_win_amount": avg_win_amount, "avg_loss_amount": avg_loss_amount,
            "winning_streak": winning_streak, "losing_streak": losing_streak,
            "recovery_factor": recovery_factor, "gain_to_pain_ratio": gain_to_pain,
            "serenity_index": serenity_index
        })
        successful_tests += 1

    except Exception as e:
        failed_tests += 1

    if combo_num % 50 == 0 or combo_num == total_combinations:
        print(f"  Combo {combo_num}/{total_combinations} | Success: {successful_tests} | Skipped(<10 trades): {skipped_low_trades} | Failed: {failed_tests}")

print(f"\nGrid search complete!")
print(f"  Successful: {successful_tests}")
print(f"  Skipped (< 10 trades): {skipped_low_trades}")
print(f"  Failed: {failed_tests}")

results_df = pd.DataFrame(grid_search_results)
if not results_df.empty:
    print(f"\nTOP 10 BY SHARPE RATIO:")
    print("=" * 90)
    top10 = results_df.nlargest(10, 'sharpe_ratio')
    for rank, (_, row) in enumerate(top10.iterrows(), 1):
        exit_label = "Middle" if row['exit_type'] == 0 else "Upper"
        print(f"  #{rank:2d} | Period={int(row['bb_period']):2d} Std={row['bb_std']:.2f} Exit={exit_label} | Sharpe={row['sharpe_ratio']:.3f} | Return={row['total_return']:.2%} | MaxDD={row['max_drawdown']:.2%} | WR={row['win_rate']:.1f}% | Trades={int(row['total_trades'])}")

In [ ]:
# TOP 5 OUT-OF-SAMPLE VALIDATION & COMPARISON TABLE

FREQ = "1D"
results_df = pd.DataFrame(grid_search_results)

if results_df.empty:
    print("No results to validate.")
else:
    print("=" * 90)
    print("TOP 5 STRATEGIES - OUT-OF-SAMPLE VALIDATION")
    print("=" * 90)
    print(f"Training Period: {train_close.index[0].date()} to {train_close.index[-1].date()}")
    print(f"Validation Period: {val_close.index[0].date()} to {val_close.index[-1].date()}")
    print("=" * 90)

    top_5 = results_df.nlargest(5, 'sharpe_ratio')
    oos_results = []
    val_vals = val_close.values.astype(float)
    val_idx = val_close.index

    for _, strategy in top_5.iterrows():
        bp = int(strategy['bb_period']); bs = float(strategy['bb_std']); et = int(strategy['exit_type'])
        
        bb_upper, bb_middle, bb_lower = talib.BBANDS(val_vals, timeperiod=bp, nbdevup=bs, nbdevdn=bs, matype=0)
        close_s = pd.Series(val_vals, index=val_idx)
        bb_upper_s = pd.Series(bb_upper, index=val_idx)
        bb_middle_s = pd.Series(bb_middle, index=val_idx)
        bb_lower_s = pd.Series(bb_lower, index=val_idx)
        
        e_raw = (close_s.shift(1) >= bb_lower_s.shift(1)) & (close_s < bb_lower_s)
        if et == 0:
            x_raw = (close_s.shift(1) <= bb_middle_s.shift(1)) & (close_s > bb_middle_s)
        else:
            x_raw = (close_s.shift(1) <= bb_upper_s.shift(1)) & (close_s > bb_upper_s)
        
        e = pd.Series(np.where(e_raw.shift(1).isna(), False, e_raw.shift(1)), index=val_idx, dtype=bool)
        x = pd.Series(np.where(x_raw.shift(1).isna(), False, x_raw.shift(1)), index=val_idx, dtype=bool)
        
        pf_val = vbt.Portfolio.from_signals(close=val_close, entries=e, exits=x,
                                            init_cash=100_000, fees=0.0005, slippage=0.0005, freq=FREQ)
        oos_sharpe = float(pf_val.sharpe_ratio(freq=FREQ))
        oos_ret = float(pf_val.total_return())
        oos_dd = float(pf_val.max_drawdown())
        trades = pf_val.trades; oos_trades = len(trades)
        oos_wr = np.nan; oos_pf = np.nan
        if oos_trades > 0:
            tr = trades.returns.values if hasattr(trades.returns, 'values') else np.array(trades.returns)
            if tr.size > 0:
                p = tr[tr > 0]; n = tr[tr < 0]
                oos_wr = (len(p)/len(tr))*100 if len(tr) > 0 else np.nan
                oos_pf = p.sum()/abs(n.sum()) if len(n) > 0 and abs(n.sum()) > 0 else np.inf
        
        exit_label = "Middle" if et == 0 else "Upper"
        oos_results.append({
            'Rank': len(oos_results)+1, 'Period': bp, 'Std': bs, 'Exit': exit_label,
            'IS_Sharpe': float(strategy['sharpe_ratio']), 'IS_Return': float(strategy['total_return']),
            'IS_MaxDD': float(strategy['max_drawdown']), 'IS_WinRate': float(strategy['win_rate']),
            'OOS_Sharpe': oos_sharpe, 'OOS_Return': oos_ret, 'OOS_MaxDD': oos_dd,
            'OOS_WinRate': oos_wr, 'OOS_Trades': oos_trades, 'OOS_ProfitFactor': oos_pf,
            'Sharpe_Degradation': ((oos_sharpe - strategy['sharpe_ratio'])/abs(strategy['sharpe_ratio'])*100) if strategy['sharpe_ratio'] != 0 else np.nan,
            'Return_Degradation': ((oos_ret - strategy['total_return'])/abs(strategy['total_return'])*100) if strategy['total_return'] != 0 else np.nan
        })

    oos_df = pd.DataFrame(oos_results)
    print("\nIS vs OOS COMPARISON:")
    print("-" * 90)
    for _, r in oos_df.iterrows():
        print(f"  #{int(r['Rank'])} | Period={int(r['Period'])} Std={r['Std']:.2f} Exit={r['Exit']}")
        print(f"       IS:  Sharpe={r['IS_Sharpe']:.3f} | Return={r['IS_Return']:.2%} | MaxDD={r['IS_MaxDD']:.2%}")
        print(f"       OOS: Sharpe={r['OOS_Sharpe']:.3f} | Return={r['OOS_Return']:.2%} | MaxDD={r['OOS_MaxDD']:.2%} | Trades={int(r['OOS_Trades'])}")
        print(f"       Degradation: Sharpe {r['Sharpe_Degradation']:+.1f}% | Return {r['Return_Degradation']:+.1f}%")
        print()

    bo = oos_df.loc[oos_df['OOS_Sharpe'].idxmax()]
    print("=" * 90)
    print(f"BEST OOS PERFORMER: Period={int(bo['Period'])} Std={bo['Std']:.2f} Exit={bo['Exit']}")
    print(f"OOS Sharpe:            {bo['OOS_Sharpe']:.3f}")
    print(f"OOS Return:            {bo['OOS_Return']:.2%}")
    print(f"OOS Max Drawdown:      {bo['OOS_MaxDD']:.2%}")
    print(f"OOS Win Rate:          {bo['OOS_WinRate']:.1f}%")
    print(f"OOS Profit Factor:     {bo['OOS_ProfitFactor']:.2f}")
    print(f"OOS Total Trades:      {int(bo['OOS_Trades'])}")
    print(f"Sharpe Degradation:    {bo['Sharpe_Degradation']:+.1f}%")
    print("=" * 90)

    # Equity curves
    bi = results_df.loc[results_df['sharpe_ratio'].idxmax()]
    bbp, bbs, bbe = int(bi['bb_period']), float(bi['bb_std']), int(bi['exit_type'])
    
    def _bb_sig(ps, period, std, exit_t):
        v = ps.values.astype(float); idx = ps.index
        bbu, bbm, bbl = talib.BBANDS(v, timeperiod=period, nbdevup=std, nbdevdn=std, matype=0)
        cs = pd.Series(v, index=idx)
        bbu_s = pd.Series(bbu, index=idx); bbm_s = pd.Series(bbm, index=idx); bbl_s = pd.Series(bbl, index=idx)
        er = (cs.shift(1) >= bbl_s.shift(1)) & (cs < bbl_s)
        if exit_t == 0:
            xr = (cs.shift(1) <= bbm_s.shift(1)) & (cs > bbm_s)
        else:
            xr = (cs.shift(1) <= bbu_s.shift(1)) & (cs > bbu_s)
        e = pd.Series(np.where(er.shift(1).isna(), False, er.shift(1)), index=idx, dtype=bool)
        x = pd.Series(np.where(xr.shift(1).isna(), False, xr.shift(1)), index=idx, dtype=bool)
        return e, x
    
    et, xt = _bb_sig(train_close, bbp, bbs, bbe)
    pft = vbt.Portfolio.from_signals(close=train_close, entries=et, exits=xt, init_cash=100_000, fees=0.0005, slippage=0.0005, freq='D')
    ev, xv = _bb_sig(val_close, bbp, bbs, bbe)
    pfv = vbt.Portfolio.from_signals(close=val_close, entries=ev, exits=xv, init_cash=100_000, fees=0.0005, slippage=0.0005, freq='D')
    
    exit_label = "Middle" if bbe == 0 else "Upper"
    fig, ax = plt.subplots(figsize=(14, 7))
    ax.plot(train_close.index, (1+pft.returns()).cumprod().values, label='Train (IS)', color='blue', linewidth=1.5, alpha=0.8)
    ax.plot(val_close.index, (1+pfv.returns()).cumprod().values, label='Validation (OOS)', color='orange', linewidth=1.5, alpha=0.8)
    ax.axvline(x=train_close.index[-1], color='red', linestyle='--', alpha=0.5, label='Train/Val Split')
    ax.set_title(f'BB(period={bbp}, std={bbs}, exit={exit_label}) - Equity Curves', fontsize=14, fontweight='bold')
    ax.set_xlabel('Date', fontsize=12); ax.set_ylabel('Cumulative Returns', fontsize=12)
    ax.grid(True, alpha=0.3); ax.legend(loc='best'); plt.tight_layout(); plt.show()

PARAMETER SENSITIVITY ANALYSIS
------------------------------

Testing how sensitive the best strategy is to parameter changes.
- Fix 2 params at best values, vary the 3rd
- Compare IS and OOS Sharpe across the range
- Green = robust, Red = fragile

In [ ]:
# PARAMETER SENSITIVITY ANALYSIS - BAR CHARTS

if results_df.empty:
    print("No results for sensitivity analysis.")
else:
    bi = results_df.loc[results_df['sharpe_ratio'].idxmax()]
    best_period = int(bi['bb_period']); best_std = float(bi['bb_std']); best_exit = int(bi['exit_type'])
    base_is_sharpe = float(bi['sharpe_ratio'])
    
    # Get base OOS sharpe
    val_vals = val_close.values.astype(float); val_idx = val_close.index
    bbu, bbm, bbl = talib.BBANDS(val_vals, timeperiod=best_period, nbdevup=best_std, nbdevdn=best_std, matype=0)
    cs = pd.Series(val_vals, index=val_idx)
    bbu_s = pd.Series(bbu, index=val_idx); bbm_s = pd.Series(bbm, index=val_idx); bbl_s = pd.Series(bbl, index=val_idx)
    er = (cs.shift(1) >= bbl_s.shift(1)) & (cs < bbl_s)
    if best_exit == 0:
        xr = (cs.shift(1) <= bbm_s.shift(1)) & (cs > bbm_s)
    else:
        xr = (cs.shift(1) <= bbu_s.shift(1)) & (cs > bbu_s)
    e = pd.Series(np.where(er.shift(1).isna(), False, er.shift(1)), index=val_idx, dtype=bool)
    x = pd.Series(np.where(xr.shift(1).isna(), False, xr.shift(1)), index=val_idx, dtype=bool)
    pf_base = vbt.Portfolio.from_signals(close=val_close, entries=e, exits=x, init_cash=100_000, fees=0.0005, slippage=0.0005, freq='D')
    base_oos_sharpe = float(pf_base.sharpe_ratio(freq='D'))
    
    exit_label = "Middle" if best_exit == 0 else "Upper"
    print(f"Best params: Period={best_period}, Std={best_std}, Exit={exit_label}")
    print(f"Base IS Sharpe: {base_is_sharpe:.3f} | Base OOS Sharpe: {base_oos_sharpe:.3f}")
    
    # Vary each parameter
    def compute_sensitivity(param_name, param_values, fixed):
        rows = []
        for val in param_values:
            if param_name == 'bb_period':
                mask = (results_df['bb_period'] == val) & (results_df['bb_std'] == fixed['bb_std']) & (results_df['exit_type'] == fixed['exit_type'])
            elif param_name == 'bb_std':
                mask = (results_df['bb_period'] == fixed['bb_period']) & (results_df['bb_std'] == val) & (results_df['exit_type'] == fixed['exit_type'])
            else:
                mask = (results_df['bb_period'] == fixed['bb_period']) & (results_df['bb_std'] == fixed['bb_std']) & (results_df['exit_type'] == val)
            subset = results_df[mask]
            if len(subset) > 0:
                is_sharpe = float(subset.iloc[0]['sharpe_ratio'])
                # Compute OOS
                bp = int(subset.iloc[0]['bb_period']); bs = float(subset.iloc[0]['bb_std']); et = int(subset.iloc[0]['exit_type'])
                bbu2, bbm2, bbl2 = talib.BBANDS(val_vals, timeperiod=bp, nbdevup=bs, nbdevdn=bs, matype=0)
                cs2 = pd.Series(val_vals, index=val_idx)
                bbu2_s = pd.Series(bbu2, index=val_idx); bbm2_s = pd.Series(bbm2, index=val_idx); bbl2_s = pd.Series(bbl2, index=val_idx)
                er2 = (cs2.shift(1) >= bbl2_s.shift(1)) & (cs2 < bbl2_s)
                if et == 0:
                    xr2 = (cs2.shift(1) <= bbm2_s.shift(1)) & (cs2 > bbm2_s)
                else:
                    xr2 = (cs2.shift(1) <= bbu2_s.shift(1)) & (cs2 > bbu2_s)
                e2 = pd.Series(np.where(er2.shift(1).isna(), False, er2.shift(1)), index=val_idx, dtype=bool)
                x2 = pd.Series(np.where(xr2.shift(1).isna(), False, xr2.shift(1)), index=val_idx, dtype=bool)
                pf2 = vbt.Portfolio.from_signals(close=val_close, entries=e2, exits=x2, init_cash=100_000, fees=0.0005, slippage=0.0005, freq='D')
                oos_sharpe = float(pf2.sharpe_ratio(freq='D'))
            else:
                is_sharpe = np.nan; oos_sharpe = np.nan
            is_delta = ((is_sharpe - base_is_sharpe) / abs(base_is_sharpe) * 100) if not np.isnan(is_sharpe) and base_is_sharpe != 0 else np.nan
            oos_delta = ((oos_sharpe - base_oos_sharpe) / abs(base_oos_sharpe) * 100) if not np.isnan(oos_sharpe) and not np.isnan(base_oos_sharpe) and base_oos_sharpe != 0 else np.nan
            rows.append({param_name: val, 'is_sharpe': is_sharpe, 'oos_sharpe': oos_sharpe, 'is_sharpe_delta': is_delta, 'oos_sharpe_delta': oos_delta})
        return pd.DataFrame(rows)
    
    fixed = {'bb_period': best_period, 'bb_std': best_std, 'exit_type': best_exit}
    pv = compute_sensitivity('bb_period', bb_period_range, fixed)
    sv = compute_sensitivity('bb_std', bb_std_range, fixed)
    ev = compute_sensitivity('exit_type', exit_type_range, fixed)
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle(f'Bollinger Band Sensitivity \u2014 Base: Period={best_period}, Std={best_std}, Exit={exit_label}', fontsize=14, fontweight='bold')
    
    for ci, (pn, var, pc) in enumerate([('BB Period', pv, 'bb_period'), ('BB Std Dev', sv, 'bb_std'), ('Exit Type', ev, 'exit_type')]):
        bv = fixed[pc]
        colors1 = ['red' if x < -10 else 'orange' if x < 0 else 'lightgreen' if x < 10 else 'darkgreen' for x in var['is_sharpe_delta']]
        axes[0, ci].bar(range(len(var)), var['is_sharpe_delta'], color=colors1, edgecolor='black', linewidth=0.5)
        axes[0, ci].set_xticks(range(len(var))); axes[0, ci].set_xticklabels(var[pc], rotation=45)
        axes[0, ci].axhline(0, color='black', linewidth=1.5)
        axes[0, ci].set_xlabel(pn, fontsize=11, fontweight='bold'); axes[0, ci].set_ylabel('Sharpe Delta %', fontsize=11, fontweight='bold')
        axes[0, ci].set_title(f'{pn} - In-Sample', fontsize=12, fontweight='bold')
        axes[0, ci].grid(axis='y', alpha=0.3)
        
        if not np.isnan(base_oos_sharpe):
            colors2 = ['red' if x < -10 else 'orange' if x < 0 else 'lightgreen' if x < 10 else 'darkgreen' for x in var['oos_sharpe_delta']]
            axes[1, ci].bar(range(len(var)), var['oos_sharpe_delta'], color=colors2, edgecolor='black', linewidth=0.5)
            axes[1, ci].set_xticks(range(len(var))); axes[1, ci].set_xticklabels(var[pc], rotation=45)
            axes[1, ci].axhline(0, color='black', linewidth=1.5)
            axes[1, ci].set_xlabel(pn, fontsize=11, fontweight='bold'); axes[1, ci].set_ylabel('Sharpe Delta %', fontsize=11, fontweight='bold')
            axes[1, ci].set_title(f'{pn} - Out-of-Sample', fontsize=12, fontweight='bold')
            axes[1, ci].grid(axis='y', alpha=0.3)
    
    plt.tight_layout(); plt.show()
    
    print("\nSENSITIVITY SUMMARY"); print("=" * 80)
    summary_data = []
    for pn, var, pc in [('BB Period', pv, 'bb_period'), ('BB Std Dev', sv, 'bb_std'), ('Exit Type', ev, 'exit_type')]:
        summary_data.append({
            'Parameter': pn,
            'IS Range': f"{var['is_sharpe'].min():.3f} - {var['is_sharpe'].max():.3f}",
            'IS Max Delta%': f"{var['is_sharpe_delta'].min():.1f}%",
            'OOS Range': f"{var['oos_sharpe'].min():.3f} - {var['oos_sharpe'].max():.3f}" if not np.isnan(base_oos_sharpe) else 'N/A',
            'OOS Max Delta%': f"{var['oos_sharpe_delta'].min():.1f}%" if not np.isnan(base_oos_sharpe) else 'N/A',
            'Sensitivity': 'HIGH' if abs(var['is_sharpe_delta'].min()) > 40 else 'LOW'
        })
    print(pd.DataFrame(summary_data).to_string(index=False))
    print("\nAnalysis Complete! LOW = Robust, HIGH = Fragile")

In [ ]:
# ================================================================
# Universal Export Cell -- Professional PDF Tearsheet + Data Files
# ================================================================
# Uses shared lib/UNIVERSAL_EXPORT_CELL_v2.py for consistent output
# across all strategy notebooks.

STRATEGY_NAME = "Bollinger_Band_MR"
PARAM_COLS = ['bb_period', 'bb_std', 'exit_type']

import os
_lib_dir = 'lib' if os.path.isdir('lib') else os.path.join('..', 'lib')
exec(open(os.path.join(_lib_dir, 'UNIVERSAL_EXPORT_CELL_v2.py'), encoding='utf-8').read())
